### `tests.ipynb` 
*Created: Sept 24, 2026* <br/>
This notebook implements a custom unit testing framework built on `Test.jl`. Unit tests are run sequentially, and their output is displayed in a table format, with one unit test per row. After all tests are completed, a summary is provided.

In [ ]:
using Test, Printf
import IJulia

In [ ]:
struct CollectedTests <: Test.AbstractTestSet
    description::String
    results::Vector{Test.Result}
end

CollectedTests(description; kwargs...) = CollectedTests(String(description), Test.Result[])

function Test.record(ts::CollectedTests, result::Test.Result)
    push!(ts.results, result)
    return result
end

function Test.record(ts::CollectedTests, child::CollectedTests)
    append!(ts.results, child.results)
    return child
end

function Test.finish(ts::CollectedTests)
    if Test.get_testset_depth() > 0
        Test.record(Test.get_testset(), ts)
    end
    return ts
end

function counts(ts::CollectedTests)
    return (passed = count(r -> r isa Test.Pass, ts.results),
            failed = count(r -> r isa Test.Fail, ts.results),
            errors = count(r -> r isa Test.Error, ts.results),
            broken = count(r -> r isa Test.Broken, ts.results),
            )
end

function status(ts::CollectedTests)
    c = counts(ts)
    c.errors > 0 && return "ERROR"
    c.failed > 0 && return "FAIL"
    c.broken > 0 && return "BROKEN"
    c.passed > 0 && return "PASS"
    return "EMPTY"
end


mutable struct LiveSuite
    title::String
    color::Bool
    rows::Vector{NamedTuple}
end

format_time(t) = @sprintf("%.2e", t)

# ── Notebook display ─────────────────────────────────────────────
function render_suite(suite; finished = false, interrupted = false)
    buffer = IOBuffer()
    out = IOContext(buffer, :color => suite.color)

    name_width = maximum(vcat(
        textwidth("Test"),
        textwidth("Total"),
        [textwidth(r.testset.description) for r in suite.rows],
    )) + 3

    result_width = 10
    time_width = 10
    table_width = name_width + result_width + time_width

    printstyled(out, suite.title, "\n"; bold = true)

    printstyled(out,
        rpad("Test", name_width),
        rpad("Result", result_width),
        lpad("Time (s)", time_width),
        "\n";
        bold = true,
    )

    for row in suite.rows
        label = status(row.testset)

        tint = label == "PASS" ? :green :
               label in ("FAIL", "ERROR") ? :red : :yellow

        print(out, rpad(row.testset.description, name_width))

        # Color only the result column.
        printstyled(out, label; color = tint)

        print(out, repeat(" ", result_width - textwidth(label)))
        println(out, lpad(format_time(row.elapsed), time_width))
    end

    println(out, repeat("─", table_width))

    if finished || interrupted
        total_time = sum(r -> r.elapsed, suite.rows; init = 0.0)

        printstyled(out,
            rpad("Total", name_width),
            rpad("", result_width),
            lpad(format_time(total_time), time_width),
            "\n";
            bold = true,
        )

        totals = [counts(r.testset) for r in suite.rows]

        passed = sum(c -> c.passed, totals; init = 0)
        failed = sum(c -> c.failed, totals; init = 0)
        errors = sum(c -> c.errors, totals; init = 0)
        broken = sum(c -> c.broken, totals; init = 0)

        empty_rows = count(
            r -> isempty(r.testset.results),
            suite.rows,
        )

        println(out, "\nAssertions/results: $passed passed · $failed failed · $errors errors",
        )

        if broken > 0
            println(out, "$broken broken/skipped assertions")
        end

        if empty_rows > 0
            println(out, "$empty_rows entries contained no assertions")
        end

        if interrupted
            println(out, "Run interrupted; totals include completed entries.")
        elseif isempty(suite.rows)
            println(out, "No tests were run.")
        elseif failed == 0 && errors == 0 &&
               broken == 0 && empty_rows == 0
            printstyled(out, "All tests passed.\n"; bold = true)
        end

        # Print native Test.jl diagnostics after the table and summary.
        for row in suite.rows
            problems = filter(r -> r isa Test.Fail || r isa Test.Error, row.testset.results)

            isempty(problems) && continue
            printstyled(out, "\n", row.testset.description, "\n"; bold = true)

            for problem in problems
                show(out, problem)
                println(out)
            end
        end
    else
        println(out, "Running tests...")
    end

    # Replace this notebook cell's output.
    IJulia.clear_output(true)
    print(stdout, String(take!(buffer)))
    flush(stdout)
    yield()

    return nothing
end

# ── Execute one named test block ─────────────────────────────────

function test!(check::Function, suite::LiveSuite, name::AbstractString)
    start = time_ns()

    ts = @testset CollectedTests "$name" begin
        check()
    end

    elapsed = (time_ns() - start) / 1e9

    push!(suite.rows, (testset = ts, elapsed = elapsed))

    render_suite(suite)
    return ts
end

# ── Execute the suite ────────────────────────────────────────────

function run_tests(body::Function; title = "Unit tests", color = true)
    Test.get_testset_depth() == 0 ||
        throw(ArgumentError(
            "Call run_tests outside an enclosing @testset.",
        ))

    suite = LiveSuite(String(title), color, NamedTuple[])
    render_suite(suite)

    try
        body(suite)
    catch
        # Manual interruption or errors outside a test! block still propagate.
        render_suite(suite; interrupted = true)
        rethrow()
    end

    render_suite(suite; finished = true)
    return suite
end

In [1]:
# # ── Example usage ────────────────────────────────────────────────
# results = run_tests(; title = "Example tests") do suite

#     test!(suite, "Addition") do
#         @test 1 + 1 == 2
#     end

#     test!(suite, "Sorting") do
#         input = [3, 1, 2]
#         output = sort(input)
#         @test output == [1, 2, 3]
#     end

#     test!(suite, "Intentional failure") do
#         @test uppercase("hello") == "hello"
#     end

#     test!(suite, "Unexpected exception") do
#         @test parse(Int, "abc") == 10
#     end

#     test!(suite, "Runs after the error") do
#         @test length("Julia") == 5
#     end

#     test!(suite, "Expected exception") do
#         @test_throws ArgumentError parse(Int, "abc")
#     end

# end;